# Stage 4 — First ML Classification (Experiment 1: Myopia vs Non-Myopia)
**Capstone: Multimodal Ocular Disease Prediction**

Pipeline position: Data Exploration → Data Cleaning + Feature ID → Normalization / Encoding + Split → **Classification** → Evaluation

This notebook:
1. Loads the Stage 3 `Data/train_set.csv` / `Data/test_set.csv` — the patient-level split, already preprocessed.
2. Trains three baseline classifiers: Logistic Regression, Random Forest, SVM.
3. Handles class imbalance via `class_weight="balanced"` (explained below — no synthetic resampling).
4. Evaluates all three on the held-out test set with accuracy, precision, recall, F1 (per-class, macro, weighted), and confusion matrices.
5. Compares the three models and identifies the best one by an appropriate metric — not raw accuracy.

No preprocessing is refit here. The imputer/scaler were already fit on training data only in Stage 3; this notebook just consumes the resulting feature columns.

## 1. Load the Stage 3 train/test sets

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

def find_path(name):
    candidates = [f"../../Data/{name}", f"Data/{name}", name]
    p = next((c for c in candidates if os.path.exists(c)), None)
    if p is None:
        raise FileNotFoundError(f"{name} not found. Checked: {candidates}")
    return p

train_path = find_path("train_set.csv")
test_path = find_path("test_set.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Loaded:", train_path, "-", train_df.shape)
print("Loaded:", test_path, "-", test_df.shape)
assert train_df.shape[0] == 5108 and test_df.shape[0] == 1284, "Shapes don't match Stage 3 output — stop and check your files."


Loaded: ../../Data/train_set.csv - (5108, 16)
Loaded: ../../Data/test_set.csv - (1284, 16)


## 2. Select the predictive feature matrix

Only the two Stage-3 model input columns are used: `Patient_Age_Scaled` and `Sex_Male`. Every diagnosis-derived / reference column carried in these files (`N, D, G, C, A, H, O, n_conditions, Myopia_OneHot_Reference, Age_Was_Missing`) is explicitly excluded here — those exist for the future multi-class experiment, not this one, and including any of them would reintroduce the leakage risk already ruled out in Stage 2.

In [2]:
FEATURE_COLS = ["Patient_Age_Scaled", "Sex_Male"]
TARGET_COL = "Myopia"

leakage_risk_cols = ["N", "D", "G", "C", "A", "H", "O", "n_conditions",
                     "Myopia_OneHot_Reference", "Age_Was_Missing", "Patient_Age", "Patient Sex", "ID"]
print("Columns available in train_set.csv:", list(train_df.columns))
print()
print("Columns actually used as model input:", FEATURE_COLS)
print("Columns present but deliberately excluded from input:",
      [c for c in leakage_risk_cols if c in train_df.columns])


Columns available in train_set.csv: ['ID', 'Patient_Age', 'Patient Sex', 'Age_Was_Missing', 'Myopia_OneHot_Reference', 'N', 'D', 'G', 'C', 'A', 'H', 'O', 'n_conditions', 'Patient_Age_Scaled', 'Sex_Male', 'Myopia']

Columns actually used as model input: ['Patient_Age_Scaled', 'Sex_Male']
Columns present but deliberately excluded from input: ['N', 'D', 'G', 'C', 'A', 'H', 'O', 'n_conditions', 'Myopia_OneHot_Reference', 'Age_Was_Missing', 'Patient_Age', 'Patient Sex', 'ID']


In [3]:
X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL].copy()
X_test = test_df[FEATURE_COLS].copy()
y_test = test_df[TARGET_COL].copy()

print("X_train:", X_train.shape, "| y_train positive rate:", y_train.mean().round(4))
print("X_test: ", X_test.shape, "| y_test positive rate:", y_test.mean().round(4))
print()
print("Any missing values in X_train?", X_train.isna().sum().sum())
print("Any missing values in X_test?", X_test.isna().sum().sum())


X_train: (5108, 2) | y_train positive rate: 0.0366
X_test:  (1284, 2) | y_test positive rate: 0.035

Any missing values in X_train? 0
Any missing values in X_test? 0


## 3. Class-imbalance handling — decision and reasoning

Myopia is ~3.7% of the training set (187 / 5,108). Three common options were considered:

| Option | Why not used here |
|---|---|
| Random oversampling / SMOTE | Would synthesize new points in a 2-feature space (one of which is a binary sex flag) — synthetic points here would mostly just duplicate/interpolate along a single real axis (age), adding little genuine information and risking an inflated, misleading test-time impression. |
| Undersampling the majority class | Would throw away real training data we already have very little of (only 2 features to begin with) — makes the "only 2 features" limitation worse, not better. |
| **Class weighting (`class_weight="balanced"`)** | **Used.** Keeps every real row exactly as collected; only changes how much each class' errors cost the optimizer during training. Standard, defensible first choice for a baseline under imbalance. |

`class_weight="balanced"` is passed to all three models below (scikit-learn computes weights as `n_samples / (n_classes * class_count)` automatically from the training labels only).

In [4]:
pos = y_train.sum()
neg = len(y_train) - pos
w_pos = len(y_train) / (2 * pos)
w_neg = len(y_train) / (2 * neg)
print(f"Training positives: {pos}, negatives: {neg}")
print(f"Balanced class weights that will be applied — Myopia=1: {w_pos:.3f}x, Myopia=0: {w_neg:.3f}x")


Training positives: 187, negatives: 4921
Balanced class weights that will be applied — Myopia=1: 13.658x, Myopia=0: 0.519x


## 4. Train the three baseline classifiers

In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", random_state=RANDOM_STATE, max_iter=1000
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "SVM (RBF kernel)": SVC(
        class_weight="balanced", kernel="rbf", probability=False, random_state=RANDOM_STATE
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")


Trained: Logistic Regression


Trained: Random Forest


Trained: SVM (RBF kernel)


## 5. Evaluate each model on the held-out test set

Metrics reported per model: accuracy, precision/recall/F1 for the Myopia (positive) class specifically, macro-averaged F1 (treats both classes equally regardless of size), weighted-averaged F1 (weights by class support), and the confusion matrix. Because the positive class is only ~3.5% of the test set, **accuracy alone is not a reliable indicator here** — a model that always predicts "Non-Myopia" would score ~96.5% accuracy while catching zero real myopia cases. Recall and macro-F1 matter more for this problem.

In [6]:
results = []
predictions = {}
conf_matrices = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    acc = accuracy_score(y_test, y_pred)
    prec_pos = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec_pos = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1_pos = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    conf_matrices[name] = cm

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision (Myopia)": prec_pos,
        "Recall (Myopia)": rec_pos,
        "F1 (Myopia)": f1_pos,
        "Precision (macro)": prec_macro,
        "Recall (macro)": rec_macro,
        "F1 (macro)": f1_macro,
        "F1 (weighted)": f1_weighted,
    })

results_df = pd.DataFrame(results).set_index("Model")
print(results_df.round(4).to_string())


                     Accuracy  Precision (Myopia)  Recall (Myopia)  F1 (Myopia)  Precision (macro)  Recall (macro)  F1 (macro)  F1 (weighted)
Model                                                                                                                                        
Logistic Regression    0.5950              0.0527           0.6222       0.0972             0.5151          0.6081      0.4181         0.7165
Random Forest          0.7477              0.0426           0.2889       0.0743             0.5050          0.5266      0.4641         0.8266
SVM (RBF kernel)       0.6012              0.0484           0.5556       0.0890             0.5111          0.5792      0.4169         0.7218


In [7]:
for name in models:
    print(f"=== {name} — confusion matrix (rows=actual, cols=predicted) ===")
    cm = conf_matrices[name]
    cm_df = pd.DataFrame(cm, index=["Actual Non-Myopia", "Actual Myopia"],
                          columns=["Pred Non-Myopia", "Pred Myopia"])
    print(cm_df)
    print()


=== Logistic Regression — confusion matrix (rows=actual, cols=predicted) ===
                   Pred Non-Myopia  Pred Myopia
Actual Non-Myopia              736          503
Actual Myopia                   17           28

=== Random Forest — confusion matrix (rows=actual, cols=predicted) ===
                   Pred Non-Myopia  Pred Myopia
Actual Non-Myopia              947          292
Actual Myopia                   32           13

=== SVM (RBF kernel) — confusion matrix (rows=actual, cols=predicted) ===
                   Pred Non-Myopia  Pred Myopia
Actual Non-Myopia              747          492
Actual Myopia                   20           25



In [8]:
for name in models:
    print(f"=== {name} — full classification report ===")
    print(classification_report(y_test, predictions[name], target_names=["Non-Myopia", "Myopia"], zero_division=0))


=== Logistic Regression — full classification report ===
              precision    recall  f1-score   support

  Non-Myopia       0.98      0.59      0.74      1239
      Myopia       0.05      0.62      0.10        45

    accuracy                           0.60      1284
   macro avg       0.52      0.61      0.42      1284
weighted avg       0.95      0.60      0.72      1284

=== Random Forest — full classification report ===
              precision    recall  f1-score   support

  Non-Myopia       0.97      0.76      0.85      1239
      Myopia       0.04      0.29      0.07        45

    accuracy                           0.75      1284
   macro avg       0.50      0.53      0.46      1284
weighted avg       0.93      0.75      0.83      1284

=== SVM (RBF kernel) — full classification report ===
              precision    recall  f1-score   support

  Non-Myopia       0.97      0.60      0.74      1239
      Myopia       0.05      0.56      0.09        45

    accuracy        

## 6. Model comparison table

In [9]:
comparison = results_df[["Accuracy", "Precision (Myopia)", "Recall (Myopia)", "F1 (Myopia)", "F1 (macro)", "F1 (weighted)"]].round(4)
comparison_sorted = comparison.sort_values("F1 (macro)", ascending=False)
print(comparison_sorted.to_string())


                     Accuracy  Precision (Myopia)  Recall (Myopia)  F1 (Myopia)  F1 (macro)  F1 (weighted)
Model                                                                                                     
Random Forest          0.7477              0.0426           0.2889       0.0743      0.4641         0.8266
Logistic Regression    0.5950              0.0527           0.6222       0.0972      0.4181         0.7165
SVM (RBF kernel)       0.6012              0.0484           0.5556       0.0890      0.4169         0.7218


## 7. Selecting the best baseline model

**Selection rule, decided before looking at the numbers:** rank by **macro-F1** first (treats catching Myopia and correctly identifying Non-Myopia as equally important, regardless of how rare Myopia is), using **recall on the Myopia class** as the tie-breaker (missing a real myopia case is generally more costly than a false alarm in a screening context). Raw accuracy is not used to choose, for the reason in Section 5.

In [10]:
best_model_name = comparison_sorted["F1 (macro)"].idxmax()
print("Best baseline model by macro-F1:", best_model_name)
print()
print(comparison_sorted.loc[[best_model_name]].to_string())


Best baseline model by macro-F1: Random Forest

               Accuracy  Precision (Myopia)  Recall (Myopia)  F1 (Myopia)  F1 (macro)  F1 (weighted)
Model                                                                                               
Random Forest    0.7477              0.0426           0.2889       0.0743      0.4641         0.8266


## 8. Save results for documentation

Numeric results are saved so the documentation step pulls the same verified numbers rather than retyping them.

In [11]:
DATA_DIR = os.path.dirname(train_path) or "."
results_out_path = os.path.join(DATA_DIR, "stage4_model_comparison.csv")
comparison_sorted.to_csv(results_out_path)
print("Saved:", results_out_path)

# Verify by re-reading
check = pd.read_csv(results_out_path, index_col=0)
print("Re-read shape:", check.shape)
assert check.shape == comparison_sorted.shape
print("Verified.")


Saved: ../../Data/stage4_model_comparison.csv
Re-read shape: (3, 6)
Verified.


## 9. Summary

| Item | Value |
|---|---|
| Features used | Patient_Age_Scaled, Sex_Male (only) |
| Target | Myopia (0/1) |
| Train / test rows | 5,108 / 1,284 |
| Train / test positive rate | 3.66% / 3.50% |
| Imbalance handling | `class_weight="balanced"` on all three models (no resampling) |
| Models trained | Logistic Regression, Random Forest, SVM (RBF kernel) |
| Selection metric | Macro-F1 (tie-break: Myopia recall) — see actual results above |
| Output file | `Data/stage4_model_comparison.csv` |

**Not started yet:** further tuning, additional models, and the fundus-image stage (Stage 6+).
